# NB_04 — Warehouse Modeling

**Project:** Nexa Commerce  
**Warehouse:** Nexa_Warehouse  
**Layer:** Warehouse  

Build the analytical Warehouse model from the validated Silver tables.

The Warehouse layer will organize the data into fact and dimension tables for downstream semantic modeling and Power BI reporting.

The modeling process will include:

- Creating dimension tables
- Creating fact tables
- Loading data from Silver tables
- Applying appropriate keys and relationships
- Validating the Warehouse model

### 1 — DimProduct

Create the DimProduct table from the validated Silver Product Master table.

In [16]:
DROP TABLE dbo.DimProduct;

CREATE TABLE dbo.DimProduct
(
    ProductKey INT NOT NULL,
    SKU  VARCHAR (50),
    ProductName VARCHAR(200),
    Category VARCHAR (100),
    Subcategory VARCHAR (100),
    SupplierID VARCHAR (50),
    UnitCostUSD DECIMAL (18,2),
    ListPriceUSD DECIMAL (18,2),
    LaunchDate DATE,
    DiscontinueDate Date,
    ActiveFlag VARCHAR (10)

);

In [17]:
INSERT INTO dbo.DimProduct
(
    ProductKey,
    SKU,
    ProductName,
    Category,
    Subcategory,
    SupplierID,
    UnitCostUSD,
    ListPriceUSD,
    LaunchDate,
    DiscontinueDate,
    ActiveFlag
)
SELECT
ROW_NUMBER () OVER (ORDER BY sku) AS ProductKey,
    sku,
    product_name,
    category,
    subcategory,
    supplier_id,
    unit_cost_usd,
    list_price_usd,
    launch_date,
    discontinue_date,
    active_flag

FROM Nexa_lakehouse.dbo.bronze_product_master


In [18]:
SELECT TOP 5 *
FROM DimProduct;

### 2 — DimLocation

Create the DimLocation table from the validated Silver Locations table.

In [31]:

CREATE TABLE dbo.DimLocations
(
    LocationKey INT NOT NULL,
    LocationId VARCHAR (50),
    LocationName VARCHAR (200),
    LocationType VARCHAR (50),
    StateOrRegion VARCHAR (50),
    ActiveFlag VARCHAR (10)

)

In [32]:
INSERT INTO dbo.DimLocations
(
    LocationKey,
    LocationId,
    LocationName,
    LocationType,
    StateOrRegion,
    ActiveFlag

)

SELECT
ROW_NUMBER () OVER (ORDER BY location_id) AS LocationKey,
    location_id,
    location_name,
    location_type,
    state_or_region,
    active_flag
FROM Nexa_Lakehouse.dbo.silver_locations

In [33]:
SELECT TOP 3 *
FROM DimLocations

### 3 — DimSupplier

Create the DimSupplier table from the validated Silver Suppliers table.

In [41]:

CREATE TABLE dbo.DimSuppliers
(
    SupplierKey INT NOT NULL,
    SupplierId VARCHAR (50),
    SupplierName VARCHAR (200),
    SupplierCountry VARCHAR (50),
    LeadTimeDays VARCHAR (50),
    ActiveFlag VARCHAR (10)


)

In [42]:
INSERT INTO dbo.DimSuppliers
(
    SupplierKey,
    SupplierId,
    SupplierName,
    SupplierCountry,
    LeadTimeDays,
    ActiveFlag

)

SELECT
ROW_NUMBER () OVER (ORDER BY supplier_id) AS SupplierKey,
    supplier_id,
    supplier_name,
    supplier_country,
    lead_time_days,
    active_flag

FROM Nexa_Lakehouse.dbo.silver_suppliers;

In [43]:
SELECT TOP 3 *
FROM DimSuppliers;

### 4 — DimCampaign

Create the DimCampaign table from the validated Silver Campaigns table.

In [52]:
DROP TABLE dbo.DimCampaigns

CREATE TABLE dbo.DimCampaigns
(
    CampaignKey INT NOT NULL,
    CampaignId VARCHAR (50),
    CampaignName VARCHAR (200),
    AdvertisingChannel VARCHAR (200), 
    Marketplace VARCHAR (50),
    StartDate DATE,
    EndDate DATE   
)

In [53]:
INSERT INTO dbo.DimCampaigns
(
    CampaignKey,
    CampaignId,
    CampaignName,
    AdvertisingChannel,
    Marketplace,
    StartDate,
    EndDate    
)
SELECT
ROW_NUMBER () OVER (ORDER BY campaign_id) AS CampaignKey,
    campaign_id,
    campaign_name,
    advertising_channel,
    marketplace,
    start_date,
    end_date
FROM Nexa_Lakehouse.dbo.silver_campaigns;

In [56]:
SELECT TOP 3 *
FROM DimCampaigns

### 5 — DimDate

Create the DimDate table for date-based analysis across the Warehouse fact tables.

In [57]:
CREATE TABLE dbo.DimDate
(
    DateKey INT NOT NULL,
    FullDate DATE NOT NULL,
    DayNumber INT,
    DayName VARCHAR (20),
    MonthNumber INT,
    MonthName VARCHAR (20),
    QuarterNumber INT,
    QuarterName VARCHAR (20),
    YearNumber INT
)

In [15]:
WITH Numbers AS 
(
    SELECT 
    ROW_NUMBER () OVER (ORDER BY a.n, b.n, c.n) -1 AS n
    FROM
    (VALUES(1),(2),(3),(4),(5),(6),(7),(8),(9),(10)) a(n),
    (VALUES(1),(2),(3),(4),(5),(6),(7),(8),(9),(10)) b(n),
    (VALUES(1),(2),(3),(4)) c(n)
 )

INSERT INTO dbo.DimDate
(
    DateKey,
    FullDate,
    DayNumber,
    DayName,
    MonthNumber,
    MonthName,
    QuarterNumber,
    QuarterName,
    YearNumber
)
SELECT 
    CAST(FORMAT(DATEADD(DAY,n, '2025-07-01'), 'yyyyMMdd') AS INT),
    DATEADD(DAY, n, '2025-07-01'),
    DAY(DATEADD(DAY, n, '2025-07-01')), 
    DATENAME(WEEKDAY, DATEADD(DAY, n, '2025-07-01')),
    MONTH(DATEADD(DAY, n, '2025-07-01')),
    DATENAME(MONTH, DATEADD(DAY, n, '2025-07-01')),
    DATEPART(QUARTER,  DATEADD(DAY, n, '2025-07-01')),
    'Q' + CAST (DATEPART(QUARTER,  DATEADD(DAY, n, '2025-07-01')) AS VARCHAR(1)),
    YEAR(DATEADD(DAY, n, '2025-07-01'))
FROM Numbers
WHERE n < 365;

In [1]:
SELECT DISTINCT
    MonthNumber,
    MonthName,
    YearNumber
FROM dbo.DimDate
ORDER BY MonthNumber;

In [ ]:
ALTER TABLE dbo.DimDate
ADD
    YearMonthKey INT,
    MonthShort VARCHAR(10);

In [ ]:
UPDATE dbo.DimDate
SET
    YearMonthKey = YEAR(FullDate) * 100 + MONTH(FullDate),
    MonthShort = LEFT(DATENAME(MONTH, FullDate), 3);

In [ ]:
SELECT TOP 3 *
FROM dbo.DimDate

### 6 — FactSales

Create the FactSales table from the validated Silver sales order and order-line tables.

In [56]:
CREATE TABLE dbo.FactSales
(

SalesKey INT NOT NULL,
Channel VARCHAR (200),
SourceOrderId VARCHAR (50),
DateKey INT NOT NULL,
ProductKey INT NOT NULL,
LineNumber INT,
Quantity INT,
Currency VARCHAR (10),
UnitPrice DECIMAL (18,2),
LineDiscount DECIMAL (18,2),
LineTax DECIMAL (18,2),
LineTotal DECIMAL (18,2),
OrderStatus VARCHAR (20)

)

In [19]:
ALTER TABLE dbo.FactSales
ADD
    UnitPriceUSD DECIMAL(18,2),
    LineDiscountUSD DECIMAL(18,2),
    LineTaxUSD DECIMAL(18,2),
    LineTotalUSD DECIMAL(18,2);

In [3]:
INSERT INTO dbo.FactSales

(

    SalesKey,
    Channel,    
    SourceOrderId,
    DateKey,
    ProductKey,
    LineNumber,
    Quantity,
    Currency,
    UnitPrice,
    LineDiscount,
    LineTax,
    LineTotal,
    OrderStatus

)

SELECT
    ROW_NUMBER () OVER (ORDER BY s.Channel, s.SourceOrderId, s.LineNumber ) AS SalesKey,
    s.Channel,    
    s.SourceOrderId,
    s.DateKey,
    s.ProductKey,
    s.LineNumber,
    s.Quantity,
    s.Currency,
    s.UnitPrice,
    s.LineDiscount,
    s.LineTax,
    s.LineTotal,
    s.OrderStatus


FROM

(
    -- Amazon--

SELECT
    'Amazon'AS Channel,
    ol.source_order_id AS SourceOrderId,
    dd.DateKey As DateKey,
    dp.ProductKey As ProductKey,
    ol.line_number AS LineNumber,
    ol.quantity As Quantity,
    ao.currency As Currency,
    ol.unit_price As UnitPrice,
    ol.line_discount AS LineDiscount,
    ol.line_tax As LineTax,
    ol.line_total As LineTotal,
    ao.order_status AS OrderStatus
    
FROM Nexa_Lakehouse.dbo.silver_amazon_order_lines ol
INNER JOIN Nexa_Lakehouse.dbo.silver_amazon_orders ao
On ol.source_order_id = ao.source_order_id
INNER JOIN dbo.DimProduct dp
ON dp.SKU = ol.sku
INNER JOIN dbo.DimDate dd
ON CAST(ao.purchase_datetime AS DATE) = dd.FullDate


UNION ALL

-- Channel_Advisor--

SELECT
    'ChannelAdvisor'AS channel,
    cl.source_order_id,
    dd.DateKey,
    dp.ProductKey,
    cl.line_number,
    cl.quantity,
    co.currency,
    cl.unit_price,
    cl.line_discount,
    cl.line_tax,
    cl.line_total,
    co.order_status

FROM Nexa_Lakehouse.dbo.silver_channeladvisor_order_lines cl
INNER JOIN Nexa_Lakehouse.dbo.silver_channeladvisor_orders co
ON cl.source_order_id = co.source_order_id
INNER JOIN dbo.DimProduct dp
ON dp.SKU = cl.sku
INNER JOIN dbo.DimDate dd
ON CAST(co.order_datetime AS DATE) = dd.FullDate

UNION ALL

--Shopify--

SELECT
    'Shopify'AS channel,
    sl.source_order_id,
    dd.DateKey,
    dp.ProductKey,
    sl.line_number,
    sl.quantity,
    so.currency,
    sl.unit_price,
    sl.line_discount,
    sl.line_tax,
    sl.line_total,
    so.order_status

FROM Nexa_Lakehouse.dbo.silver_shopify_order_lines sl
INNER JOIN Nexa_Lakehouse.dbo.silver_shopify_orders so
ON sl.source_order_id = so.source_order_id
INNER JOIN dbo.DimProduct dp
ON dp.SKU = sl.sku
INNER JOIN dbo.DimDate dd
ON CAST(so.order_datetime AS DATE) = dd.FullDate

UNION ALL

--Walmart--

SELECT
    'Walmart'AS channel,
    wl.source_order_id,
    dd.DateKey,
    dp.ProductKey,
    wl.line_number,
    wl.quantity,
    wo.currency,
    wl.unit_price,
    wl.line_discount,
    wl.line_tax,
    wl.line_total,
    wo.order_status

FROM Nexa_Lakehouse.dbo.silver_walmart_order_lines wl
INNER JOIN Nexa_Lakehouse.dbo.silver_walmart_orders wo
ON wl.source_order_id = wo.source_order_id
INNER JOIN dbo.DimProduct dp
ON dp.SKU = wl.sku
INNER JOIN dbo.DimDate dd
ON CAST(wo.order_datetime AS DATE) = dd.FullDate

)s

In [20]:
UPDATE dbo.FactSales
SET
    UnitPriceUSD =
        CASE
            WHEN Currency = 'CAD' THEN ROUND(UnitPrice * 0.73, 2)
            ELSE UnitPrice
        END,

    LineDiscountUSD =
        CASE
            WHEN Currency = 'CAD' THEN ROUND(LineDiscount * 0.73, 2)
            ELSE LineDiscount
        END,

    LineTaxUSD =
        CASE
            WHEN Currency = 'CAD' THEN ROUND(LineTax * 0.73, 2)
            ELSE LineTax
        END,

    LineTotalUSD =
        CASE
            WHEN Currency = 'CAD' THEN ROUND(LineTotal * 0.73, 2)
            ELSE LineTotal
        END;

In [21]:
SELECT
    Currency,
    COUNT(*) AS CurrencyRows,
    SUM(LineTotal) AS SourceValue,
    SUM(LineTotalUSD) AS USDValue
FROM dbo.FactSales
GROUP BY Currency
ORDER BY Currency;

In [22]:
SELECT TOP 3 *
FROM dbo.FactSales

In [23]:
SELECT
    COUNT(*) AS TotalRows,
    SUM(CASE WHEN LineTotalUSD IS NULL THEN 1 ELSE 0 END) AS MissingUSD,
    SUM(LineTotalUSD) AS TotalSalesUSD
FROM dbo.FactSales;

In [7]:
SELECT 
Channel,
COUNT (*)
FROM dbo.FactSales
GROUP BY Channel

In [ ]:
SELECT 
MAX(snapshot_date),
ending_on_hand
FROM Nexa_Lakehouse.dbo.silver_inventory_daily
GROUP BY 


DatawarehouseExecuteSqlError: Column 'Nexa_Lakehouse.dbo.silver_inventory_daily.ending_on_hand' is invalid in the select list because it is not contained in either an aggregate function or the GROUP BY clause. Data warehouse trace id: 6211dbfb-ecdb-40ef-be7b-803295116e4f

### 7 — FactInventory

Create the FactInventory table from the validated Silver Inventory table.

In [5]:
CREATE TABLE dbo.FactInventory
(
    InventoryKey INT NOT NULL,
    SnapShotKey INT NOT NULL,
    LocationKey INT NOT NULL,
    ProductKey INT NOT NULL,
    BeginningOnHand INT NOT NULL,
    ReceiptsQty INT,
    SalesQty INT,
    ReturnQty INT,
    AdjustmentQty INT,
    EndingOnHand INT NOT NULL,
    ReservedQty INT NOT NULL
)

In [6]:
INSERT INTO dbo.FactInventory
(
    InventoryKey,
    SnapShotKey,
    LocationKey,
    ProductKey,
    BeginningOnHand,
    ReceiptsQty,
    SalesQty,
    ReturnQty,
    AdjustmentQty,
    EndingOnHand,
    ReservedQty
)

SELECT
ROW_NUMBER () OVER (ORDER BY d.SnapShotKey, d.LocationKey, d.ProductKey ) As InventoryKey,
    d.SnapShotKey,
    d.LocationKey,
    d.ProductKey,
    d.BeginningOnHand,
    d.ReceiptsQty,
    d.SalesQty,
    d.ReturnQty,
    d.AdjustmentQty,
    d.EndingOnHand,
    d.ReservedQty

FROM
(
SELECT
    dd.DateKey AS SnapShotKey,
    dl.LocationKey AS LocationKey,
    dp.ProductKey AS ProductKey,
    id.beginning_on_hand AS BeginningOnHand ,
    id.receipts_qty AS ReceiptsQty,
    id.sales_qty AS SalesQty,
    id.returns_qty As ReturnQty,
    id.adjustments_qty AS AdjustmentQty,
    id.ending_on_hand AS EndingOnHand,
    id.reserved_qty AS ReservedQty

FROM Nexa_Lakehouse.dbo.silver_inventory_daily id
INNER JOIN dbo.DimDate dd
ON id.snapshot_date = dd.FullDate
INNER JOIN dbo.DimLocations dl 
ON id.location_id = dl.LocationId
INNER JOIN dbo.DimProduct dp 
ON id.sku = dp.SKU
) d;

In [17]:
SELECT COUNT(InventoryKey)
FROM FactInventory

In [ ]:
SELECT
    COUNT(*) AS SilverRows,
    (
        SELECT COUNT(*)
        FROM dbo.FactInventory
    ) AS WarehouseRows;

### 8 — FactRefund

Create the FactRefund table from the validated Silver Refund Events table.

In [15]:
CREATE TABLE dbo.FactRefund
(
    RefundKey INT NOT NULL,
    RefundId VARCHAR (50),
    SourceChannel VARCHAR (20),
    SourceOrderId VARCHAR (50),
    RefundDateKey INT NOT NULL,
    ProductKey INT NULL,
    QuantityRefunded INT,
    RefundAmount DECIMAL (18,2),
    Currency VARCHAR (10),
    RefundReason VARCHAR (200)

)

In [23]:
INSERT INTO dbo.FactRefund
(
    RefundKey,
    RefundId,
    SourceChannel,
    SourceOrderId,
    RefundDateKey,
    ProductKey,
    QuantityRefunded,
    RefundAmount,
    Currency,
    RefundReason
)
SELECT
    ROW_NUMBER () OVER (ORDER BY refund_id, source_channel) AS RefundKey,
    re.refund_id,
    re.source_channel,
    re.source_order_id,
    dd.DateKey AS RefundDateKey,
    dp.ProductKey,
    re.quantity_refunded,
    re.refund_amount,
    re.currency,
    re.refund_reason
FROM Nexa_Lakehouse.dbo.silver_refund_events re
INNER JOIN dbo.DimProduct dp
ON re.sku = dp.SKU
INNER JOIN dbo.DimDate dd
ON CAST (re.refund_date AS DATE) = dd.FullDate; 



In [24]:
SELECT COUNT (*)
FROM FactRefund

In [3]:
UPDATE dbo.FactRefund
SET SourceChannel = 'Shopify'
WHERE SourceChannel = 'shopify';

### 9 — FactPurchaseOrder

Create the FactPurchaseOrder table from the validated Silver Purchase Order tables.

In [52]:
CREATE TABLE dbo.FactPurchase
(
    PurchaseKey INT NOT NULL,
    POId VARCHAR(50),
    PODateKey INT NOT NULL,
    SupplierKey INT NOT NULL,
    DestLocationKey INT NOT NULL,
    ExpectedDeliveryDateKey INT,
    ActualDeliveryDateKey INT,
    ProductKey INT NOT NULL,
    LineNumber INT,
    QuantityOrdered INT,
    QuantityReceived INT,
    UnitCost DECIMAL(18,2),
    Status VARCHAR(20),
    Currency VARCHAR(10)
);

In [56]:
INSERT INTO dbo.FactPurchase
(
    PurchaseKey,
    POId,
    PODateKey,
    SupplierKey,
    DestLocationKey,
    ExpectedDeliveryDateKey,
    ActualDeliveryDateKey,
    ProductKey,
    LineNumber,
    QuantityOrdered,
    QuantityReceived,
    UnitCost,
    Status,
    Currency
)
SELECT 
    ROW_NUMBER () OVER (ORDER BY ol.po_id) AS POKey,
    ol.po_id,
    dd.DateKey AS PODateKey,
    ds.SupplierKey AS SupplierKey,
    dl.LocationKey AS DestLocationKey,
    dd.DateKey AS ExpectedDeliveryDateKey,
    dd.DateKey As ActualDeliveryDateKey,
    dp.ProductKey,
    ol.line_number AS LineNumber,
    ol.quantity_ordered,
    ol.quantity_received,
    ol.unit_cost,
    po.status,
    po.currency

FROM Nexa_Lakehouse.dbo.silver_purchase_order_lines ol
INNER JOIN Nexa_Lakehouse.dbo.silver_purchase_orders po
    ON ol.po_id = po.po_id
INNER JOIN dbo.DimDate dd 
    ON po.po_date = dd.FullDate
INNER JOIN dbo.DimSuppliers ds 
    ON po.supplier_id = ds.SupplierId
INNER JOIN DimLocations dl
    ON po.destination_location_id = dl.LocationId
INNER JOIN  DimProduct dp
    ON ol.sku = dp.SKU

In [59]:
SELECT TOP 20 *
FROM FactPurchase;

### 9 — FactAdvertising

Create the FactAdvertising table from the validated Silver Google Ads and Walmart Connect advertising tables.

In [63]:
SELECT TOP 10 *
FROM Nexa_Lakehouse.dbo.silver_google_ads_campaign_daily

In [64]:
SELECT TOP 10 *
FROM Nexa_Lakehouse.dbo.silver_walmart_connect_weekly

In [2]:
SELECT TOP 3 *
FROM DimCampaigns

In [65]:
CREATE TABLE dbo.FactAdvertising
(
AdvertisingKey INT NOT NULL,
Source VARCHAR (50),
CampaignKey INT NOT NULL,
DateKey INT,
PeriodType VARCHAR (20),
Device VARCHAR (20),
Impressions INT,
Clicks INT,
Spend DECIMAL (18,2),
Conversion INT,
AttributedRevenue DECIMAL (18,2)


)

In [81]:
INSERT INTO dbo.FactAdvertising
(
    AdvertisingKey,
    Source,
    CampaignKey,
    DateKey,
    PeriodType,
    Device,
    Impressions,
    Clicks,
    Spend,
    Conversion,
    AttributedRevenue
)

SELECT
    ROW_NUMBER () OVER (ORDER BY Source, CampaignKey, Device) AS AdvertisingKey,
    Source,
    CampaignKey,
    DateKey,
    PeriodType,
    Device,
    Impressions,
    Clicks,
    Spend,
    Conversion,
    AttributedRevenue


FROM
(
-- Google Ads
SELECT 
    
    'Google Ads' AS Source,
    dc.CampaignKey AS CampaignKey,
    dd.DateKey,
    'Daily' AS PeriodType,
    gc.device AS Device,
    gc.impressions AS Impressions,
    gc.clicks As Clicks,
    gc.spend AS Spend,
    gc.conversions AS Conversion,
    gc.attributed_revenue AS AttributedRevenue

FROM Nexa_Lakehouse.dbo.silver_google_ads_campaign_daily gc 
INNER JOIN dbo.DimDate dd 
    ON gc.activity_date = dd.FullDate 
INNER JOIN dbo.DimCampaigns dc
    ON gc.campaign_id = dc.CampaignId


UNION ALL

-- Walmart Connect

SELECT 
    'Walmart' AS Source,
    dc.CampaignKey AS CampaignKey,
    dd.DateKey,
    'Weekly' AS PeriodType,
    'N/A' AS Device,
    wc.impressions AS Impressions,
    wc.clicks As Clicks,
    wc.spend AS Spend,
    wc.conversions AS Conversion,
    wc.attributed_revenue AS AttributedRevenue


FROM Nexa_Lakehouse.dbo.silver_walmart_connect_weekly wc
INNER JOIN DimDate dd
    ON wc.week_start_date = dd.FullDate
INNER JOIN dbo.DimCampaigns dc
    ON wc.campaign_id = dc.CampaignId

) AS Advertising;

In [83]:
SELECT
Source,
COUNT(AdvertisingKey)
FROM FactAdvertising
GROUP BY Source;

### 10 — Warehouse Validation

Validate the Warehouse model using row counts, dimension key coverage, fact grain, measure integrity, and date coverage.

### 10.1 — Fact and Silver Validation

Validate Warehouse fact tables against their corresponding Silver source tables by comparing row counts and confirming that the expected source records are represented in the Warehouse.

In [24]:

--Amazon

SELECT 
    'Amazon' AS Channel,
    COUNT (*) AS SilverCount,
(

SELECT 
    COUNT(*)
    FROM dbo.FactSales
    WHERE Channel = 'Amazon'

) AS FactCount

FROM Nexa_Lakehouse.dbo.silver_amazon_order_lines

UNION ALL 

--Channel Advisor

SELECT 
    'Channel Advisor' AS Channel,
    COUNT (*) AS SilverCount,
(

SELECT 
    COUNT(*)
    FROM dbo.FactSales
    WHERE Channel = 'ChannelAdvisor'

) AS FactCount

FROM Nexa_Lakehouse.dbo.silver_channeladvisor_order_lines


UNION ALL 

--Walmart

SELECT 
    'Walmart' AS Channel,
    COUNT (*) AS SilverCount,
(

SELECT 
    COUNT(*)
    FROM dbo.FactSales
    WHERE Channel = 'Walmart'

) AS FactCount

FROM Nexa_Lakehouse.dbo.silver_walmart_order_lines

UNION ALL

--Shopify

SELECT 
    'Shopify' AS Channel,
    COUNT (*) AS SilverCount,
(

SELECT 
    COUNT(*)
    FROM dbo.FactSales
    WHERE Channel = 'Shopify'

) AS FactCount

FROM Nexa_Lakehouse.dbo.silver_shopify_order_lines

UNION ALL

--Google Ads

SELECT 
    'Google Ads' AS Channel,
    COUNT (*) AS SilverCount,
(

SELECT 
    COUNT(*)
    FROM dbo.FactAdvertising
    WHERE Source = 'Google Ads'

) AS FactCount

FROM Nexa_Lakehouse.dbo.silver_google_ads_campaign_daily



### 10.2 — Dimension Key Validation

Validate that all foreign keys stored in the Warehouse fact tables have matching records in their corresponding dimension tables.

In [40]:
--DimProduct

SELECT 
    'DimProduct to FactSales' AS ValidationLayer,
    COUNT(*) AS MissingProductKeys
FROM dbo.FactSales fs
LEFT JOIN dbo.DimProduct dp
    ON fs.ProductKey = dp.ProductKey    
WHERE dp.ProductKey IS NULL

UNION ALL

--DimSuppliers

SELECT 
    'DimSuppliers to FactPurchase' AS ValidationLayer,
    COUNT(*) AS MissingLocationKey
FROM dbo.FactPurchase fp
LEFT JOIN dbo.DimSuppliers ds
    ON fp.SupplierKey = ds.SupplierKey
WHERE ds.SupplierKey IS NULL;


### 10.3 — Fact Grain Validation

Validate that each Warehouse fact table maintains its defined grain and does not contain unexpected duplicate records.

In [47]:
SELECT
    SourceOrderId,
    LineNumber,
    COUNT(*) AS Row_Count
FROM dbo.FactSales
GROUP BY
    SourceOrderId,
    LineNumber
HAVING COUNT(*) > 1;

### 10.4 — Measure Validation

Validate key measures in the Warehouse against the corresponding Silver data to confirm that values were loaded correctly and no unexpected negative or missing values were introduced.

In [51]:
SELECT
    COUNT(*) AS InvalidRows
FROM dbo.FactSales 
WHERE Quantity < 0
   OR UnitPrice < 0
   OR LineDiscount < 0
   OR LineTax < 0
   OR LineTotal <0 
   OR ABS(
        (Quantity * UnitPrice - LineDiscount + LineTax) - LineTotal
      ) > 0.01;

### 10.5 — Date Coverage Validation

Validate that Warehouse fact dates fall within the defined reporting period.

In [53]:
SELECT
    COUNT(*) AS MissingDateKeys
FROM dbo.FactSales fs
LEFT JOIN DimDate dd 
    ON fs.DateKey = dd.DateKey
WHERE dd.DateKey IS NULL 

In [ ]:
SELECT
    OrderStatus,
    SUM(LineTotal) AS TotalLineValue
FROM dbo.FactSales
GROUP BY OrderStatus
ORDER BY OrderStatus;

In [ ]:
SELECT TOP 10
    ol.source_order_id,
    sku,
    currency,
    unit_price,
    line_discount,
    line_tax,
    line_total
FROM Nexa_Lakehouse.dbo.silver_shopify_order_lines ol 
INNER JOIN Nexa_Lakehouse.dbo.silver_shopify_orders so
    ON ol.source_order_id = so.source_order_id
WHERE currency = 'CAD';

## 11 — DimChannel

Create and populate the channel dimension.

This dimension will provide a consistent channel reference for Sales and Refund analysis.

In [24]:
CREATE TABLE dbo.DimChannel
(
ChannelKey INT NOT NULL,
ChannelName VARCHAR (20)

);

In [25]:
INSERT INTO dbo.DimChannel
(
ChannelKey,
ChannelName 

)
SELECT
ROW_NUMBER () OVER (ORDER BY Channel) AS ChannelKey,
Channel

FROM
(
SELECT DISTINCT Channel
FROM dbo.FactSales

) AS chanels;

In [26]:
SELECT DISTINCT ChannelName
FROM DimChannel